In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from astrobridge.paper_pairing.core import CrossmatchBundle
from astrobridge.paper_pairing.filters import (
    MaxObjectsPerPaperFilter,
    MaxPapersPerObjectFilter,
    ObjectNameInTitleOrAbstractFilter,
    TitleKeywordFilter,
)
from astrobridge.paper_pairing.augmenters import AliasAugmenter

In [3]:
bundle = CrossmatchBundle.load("../data/paper_crossmatch_bundles/mmu_desi_edr_sv3")
print(bundle.summary())

CrossmatchBundle Summary
  hf_objects: 84555 rows, columns=['object_id', 'ra', 'dec']
  simbad_objects: 82600 rows, columns=['main_id', 'coo_bibcode']
  ads_papers: 10943 rows, columns=['bibcode', 'paper_title', 'abstract', 'doi', 'keyword', 'preprint_url']
  relationships: 358960 rows, columns=['hf_id', 'simbad_main_id', 'bibcode']


In [4]:
augmenter = AliasAugmenter()
augmenter.augment(bundle, inplace=True)
print(bundle.summary())

SIMBAD TAP alias chunks:   0%|          | 0/5 [00:00<?, ?it/s]

CrossmatchBundle Summary
  hf_objects: 84555 rows, columns=['object_id', 'ra', 'dec']
  simbad_objects: 82600 rows, columns=['main_id', 'coo_bibcode', 'aliases']
  ads_papers: 10943 rows, columns=['bibcode', 'paper_title', 'abstract', 'doi', 'keyword', 'preprint_url']
  relationships: 358960 rows, columns=['hf_id', 'simbad_main_id', 'bibcode']


In [5]:
bundle.save("../data/paper_crossmatch_bundles/mmu_desi_edr_sv3_augmented")

PosixPath('../data/paper_crossmatch_bundles/mmu_desi_edr_sv3_augmented')

In [6]:
all_aliases = []
df = bundle.simbad_objects

for i in range(len(df)):
    all_aliases.extend(df['aliases'].values[i])

values, counts = np.unique(all_aliases, return_counts=True)
assert all(counts == 1), "Found a repeated alias for two distinct objects"